# EDA — understanding the two source datasets

**Goal:** understand the *shape and dirtiness* of the TiC (payer) and HPT (hospital) extracts before any matching. Run top to bottom; only needs `pandas`.

The brief: combine a **payer** Transparency-in-Coverage extract with a **hospital** Price-Transparency extract into one unified per-rate table, identify which records match across the two, and quantify where the rates agree or disagree.

The sample is deliberately tiny and controlled: **3 hospitals × 3 billing codes × 3 payers**. Everything hard about this problem is already visible at this scale.

In [1]:
# --- setup: find the repo root and make `core` importable from anywhere ---
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "core").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

TIC_PATH = ROOT / "data" / "tic_extract_20250213.csv"
HPT_PATH = ROOT / "data" / "hpt_extract_20250213.csv"
print("repo root:", ROOT)
print("TiC:", TIC_PATH.exists(), "| HPT:", HPT_PATH.exists())

repo root: /Users/u0933981/Desktop/Work/serif_takehome
TiC: True | HPT: True


In [2]:
# Load both files as strings (preserve leading zeros in codes/EINs)
tic = pd.read_csv(TIC_PATH, dtype=str)
hpt = pd.read_csv(HPT_PATH, dtype=str)
print("TiC (payer) :", tic.shape)
print("HPT (hospital):", hpt.shape)

TiC (payer) : (222, 17)
HPT (hospital): (2950, 22)


## 1. Two different schemas for the same underlying reality

- **TiC (Transparency in Coverage)** is published by *insurers*. One row = a negotiated rate for a (code, provider-group, network) under a commercial plan. Here it's the three big national PPOs.
- **HPT (Hospital Price Transparency)** is published by *hospitals*. One row = a rate the hospital accepts for a code under a named payer **plan**, plus cash/gross prices.

They describe the same negotiated prices from opposite sides of the contract — but the column names, identifiers, and granularity differ. That mismatch *is* the assignment.

In [3]:
# TiC columns and a transposed sample row
print("TiC columns:\n", list(tic.columns))
tic.head(2).T

TiC columns:
 ['payer', 'network_name', 'network_id', 'network_year_month', 'network_region', 'code', 'code_type', 'ein', 'taxonomy_filtered_npi_list', 'modifier_list', 'billing_class', 'place_of_service_list', 'negotiation_type', 'arrangement', 'rate', 'cms_baseline_schedule', 'cms_baseline_rate']


,0,1
payer,unitedhealthcare,unitedhealthcare
network_name,choice-plus,choice-plus
network_id,592bc118-0dac-4f38-949c-11dc9b3a3879,592bc118-0dac-4f38-949c-11dc9b3a3879
network_year_month,202501,202501
network_region,USA,USA
code,872,99283
code_type,MS-DRG,CPT
ein,131740114,131624096
taxonomy_filtered_npi_list,"1003990763,1023202793,1063525152,1063606739,10...","1003255670,1245759711,1487026522,1598095267,16..."
modifier_list,NaN,NaN


In [4]:
# The join-relevant TiC fields, at a glance
for col in ["payer", "code", "code_type", "billing_class", "negotiation_type", "arrangement"]:
    print(f"--- {col} ---")
    print(tic[col].value_counts(dropna=False).to_string(), "\n")

--- payer ---
payer
aetna                133
unitedhealthcare      45
cigna-corporation     44 

--- code ---
code
43239    103
99283    100
872       19 

--- code_type ---
code_type
CPT       203
MS-DRG     19 

--- billing_class ---
billing_class
professional     177
institutional     45 

--- negotiation_type ---
negotiation_type
negotiated      169
fee schedule     50
percentage        3 

--- arrangement ---
arrangement
ffs    222 



**Read this:** TiC already carries `billing_class` (institutional vs professional) — HPT will *not*. `negotiation_type` is mostly `negotiated`/`fee schedule` with a few `percentage` rows (those encode "X% of a baseline", not a dollar — they'll be dropped). `arrangement` is all `ffs` here, but at scale per-diem/bundle/capitation appear and change what the number *means*.

In [5]:
# HPT columns and a transposed sample row
print("HPT columns:\n", list(hpt.columns))
hpt.head(2).T

HPT columns:
 ['source_file_name', 'hospital_id', 'hospital_name', 'last_updated_on', 'hospital_state', 'license_number', 'payer_name', 'plan_name', 'code_type', 'raw_code', 'description', 'setting', 'modifiers', 'standard_charge_gross', 'standard_charge_discounted_cash', 'standard_charge_negotiated_dollar', 'standard_charge_negotiated_percentage', 'standard_charge_min', 'standard_charge_max', 'standard_charge_methodology', 'additional_payer_notes', 'additional_generic_notes']


,0,1
source_file_name,13-1740114_montefiore-medical-center_standardc...,13-1740114_montefiore-medical-center_standardc...
hospital_id,62915ae8-8d64-4e2f-b05f-b18edde57a3d,62915ae8-8d64-4e2f-b05f-b18edde57a3d
hospital_name,Montefiore Medical Center,Montefiore Medical Center
last_updated_on,2024-07-01,2024-07-01
hospital_state,NY,NY
license_number,13-1740114,13-1740114
payer_name,Aetna,HealthFirst
plan_name,Medicare,Commercial Enrollees
code_type,CPT,CPT
raw_code,99283,43239


In [6]:
# HPT key fields
print("code_type:\n", hpt.code_type.value_counts(dropna=False).to_string(), "\n")
print("setting:\n", hpt.setting.value_counts(dropna=False).to_string(), "\n")
print("distinct payer_name strings:", hpt.payer_name.nunique())
print("distinct plan_name strings :", hpt.plan_name.nunique())

code_type:
 code_type
CPT       2162
LOCAL      410
MS-DRG     378 

setting:
 setting
both          2208
inpatient      408
outpatient     334 

distinct payer_name strings: 73
distinct plan_name strings : 538


## 2. The three codes

- **43239** — CPT/HCPCS: upper-GI endoscopy with biopsy (a *procedure*).
- **99283** — CPT/HCPCS: emergency-department visit, level 3 (a *visit*).
- **872** — MS-DRG: "septicemia/severe sepsis without MV >96h, without MCC" (an *inpatient bundle*).

The DRG is the structurally different one: it's a single bundled inpatient payment with **no professional side**, and it runs an order of magnitude larger than the CPTs.

In [7]:
from core.normalizers import normalize_code
tic["_code"] = tic.apply(lambda r: normalize_code(r["code"], r["code_type"]), axis=1)
hpt["_code"] = hpt.apply(lambda r: normalize_code(r["raw_code"], r["code_type"]), axis=1)
print("TiC normalized codes:", sorted(tic._code.dropna().unique()))
print("HPT normalized codes:", sorted(hpt._code.dropna().unique()))
# DRG 872 is written differently across hospitals -> normalization matters:
print("\nraw_code spellings for DRG 872 in HPT:",
      sorted(hpt.loc[hpt._code=="872", "raw_code"].unique()))

TiC normalized codes: ['43239', '872', '99283']
HPT normalized codes: ['43239', '872', '99283']

raw_code spellings for DRG 872 in HPT: ['872', 'MS-DRG 872']


## 3. The hospital-identity puzzle (the single highest-leverage cleaning step)

To join a hospital's HPT file to the payer's TiC rows you need a shared key. The natural one is the **federal EIN**. The trap: only one of three hospitals actually puts its EIN in `license_number`.

In [8]:
from core.normalizers import normalize_ein, ein_from_hpt_source_file
id_table = (hpt.groupby("hospital_name")
              .agg(license_number=("license_number", "first"),
                   source_file=("source_file_name", "first"))
              .reset_index())
id_table["ein_from_license"]  = id_table.license_number.apply(normalize_ein)
id_table["ein_from_filename"] = id_table.source_file.apply(ein_from_hpt_source_file)
id_table[["hospital_name","license_number","ein_from_license","ein_from_filename"]]

,hospital_name,license_number,ein_from_license,ein_from_filename
0,Montefiore Medical Center,13-1740114,131740114,131740114
1,NYU Langone,7002053H,007002053,133971298
2,The Mount Sinai Hospital,330024,000330024,131624096


**Read this:** `license_number` is a NY state operating certificate / DOH facility id for two of the three hospitals — *not* an EIN. The federal EIN is reliably encoded only in the **filename prefix** (CMS HPT v2 requires this). Key on the filename EIN and recall is 3/3 hospitals; key on `license_number` and it's 1/3.

## 4. Payers — a clean trio here, an exploding long tail at scale

In [9]:
from core.normalizers import load_payer_aliases, normalize_payer
aliases = load_payer_aliases(ROOT / "core" / "payer_aliases.json")
hpt["_payer"] = hpt.payer_name.apply(lambda p: normalize_payer(p, aliases))
resolved = hpt._payer.notna().mean()
print(f"HPT rows whose payer string resolves to a canonical payer: {resolved:.0%}")
print("\nTop HPT payer_name strings (note the long tail that does NOT resolve):")
print(hpt.payer_name.value_counts().head(12).to_string())

HPT rows whose payer string resolves to a canonical payer: 13%

Top HPT payer_name strings (note the long tail that does NOT resolve):
payer_name
hip            392
medicare       336
multiplan      276
bcbs           273
healthfirst    137
aetna          113
cigna          102
fidelis         95
uhc             89
ghi             83
somos           79
MetroPlus       79


**Read this:** ~33% of HPT rows resolve here; the rest are a long tail (Emblem, Fidelis, Healthfirst, MetroPlus, Centivo, …) that aren't among the three payers in the TiC sample. At national scale this long tail is the *dominant* filter — payer-name resolution, not compute, sets the match rate.

## 5. Billing class: TiC has it, HPT must be inferred

A CPT like 43239 is billed twice — once by the *facility* (institutional) and once by the *physician* (professional) — at very different rates. TiC labels which is which; HPT does not, so we infer it from a description prefix (`HC ` = hospital/institutional, `PR ` = professional) or the setting.

In [10]:
d = hpt.description.fillna("").str.upper()
print("HPT description prefixes:  HC ->", int(d.str.startswith("HC ").sum()),
      "| PR ->", int(d.str.startswith("PR ").sum()),
      "| neither ->", int((~d.str.startswith(("HC ","PR "))).sum()))
from core.normalizers import infer_hpt_billing_class
ex = hpt[hpt._code=="43239"].head(6)
for _, r in ex.iterrows():
    print(f"{infer_hpt_billing_class(r.code_type, r.description, r.setting):13} <- {str(r.description)[:45]!r}")

HPT description prefixes:  HC -> 1350 | PR -> 176 | neither -> 1424
unknown       <- 'EGD BIOPSY SINGLE/MULTIPLE'
unknown       <- 'UPPER GI ENDOSCOPY BIOPSY'
unknown       <- 'EGD BIOPSY SINGLE/MULTIPLE'
professional  <- 'PR EDG TRANSORAL BIOPSY SINGLE/MULTIPLE'
professional  <- 'PR EDG TRANSORAL BIOPSY SINGLE/MULTIPLE'
professional  <- 'PR EDG TRANSORAL BIOPSY SINGLE/MULTIPLE'


**Read this:** the `HC`/`PR` prefix is a *weak* signal — many rows have neither, so they fall to `unknown`. Section 8 shows a real case ($6,438) where this heuristic misroutes a genuinely-institutional rate.

## 6. Dirtiness that has to be handled before matching

In [11]:
hpt["_dollar"] = pd.to_numeric(hpt.standard_charge_negotiated_dollar, errors="coerce")
print("TiC negotiation_type='percentage' rows (encode % of baseline, not $):",
      int((tic.negotiation_type.str.lower()=="percentage").sum()))
print("HPT code_type='LOCAL' chargemaster rows (CPT-shaped collisions)      :",
      int((hpt.code_type.str.lower()=="local").sum()))
print("HPT rows with no negotiated dollar (percent-of-charges only, etc.)   :",
      int(hpt._dollar.isna().sum()))

TiC negotiation_type='percentage' rows (encode % of baseline, not $): 3
HPT code_type='LOCAL' chargemaster rows (CPT-shaped collisions)      : 410
HPT rows with no negotiated dollar (percent-of-charges only, etc.)   : 426


## 7. Brief example #1 — the $6,438 match (it IS in the data)

> *"The UHC rate for 43239 is listed at 6438 in their hospital file. When you look at the UHC payer dataset, the same rate appears."*

It reproduces at **Mount Sinai**, as an **institutional** rate, on both sides.

In [12]:
hpt["_ein"] = hpt.source_file_name.apply(ein_from_hpt_source_file)
tic["_ein"] = tic.ein.apply(normalize_ein)
tic["_payer"] = tic.payer.apply(lambda p: normalize_payer(p, aliases))
ms_hpt = hpt[(hpt._ein=="131624096") & (hpt._code=="43239") & (hpt._payer=="unitedhealthcare")].copy()
ms_tic = tic[(tic._ein=="131624096") & (tic._code=="43239") & (tic._payer=="unitedhealthcare")].copy()
print("HPT Mount Sinai UHC 43239 negotiated dollars:",
      sorted(set(ms_hpt._dollar.dropna().round(2))))
ms_tic["_r"] = pd.to_numeric(ms_tic.rate, errors="coerce")
for bc, g in ms_tic.groupby("billing_class"):
    print(f"TiC Mount Sinai UHC 43239 {bc}: {sorted(set(g._r.dropna().round(2)))}")

HPT Mount Sinai UHC 43239 negotiated dollars: [1048.28, 6438.0]
TiC Mount Sinai UHC 43239 institutional: [1532.0, 2670.0, 4608.0, 6438.0]
TiC Mount Sinai UHC 43239 professional: [270.93, 311.55, 514.22, 862.79]


**Read this:** `$6,438` is in **both** files (HPT, and TiC *institutional*). But note the HPT `$6,438` row's description and how billing-class inference treats it — that's why the key-level join splits it, and why the pipeline adds a rate-value corroboration pass (see `pipeline.ipynb`).

## 8. Brief example #2 — Aetna DRG 872 (the expected gap)

> *"Montefiore lists Aetna PPO DRG 872 at 29259.18. Do you see that in the Aetna TiC extract? (You will not.)"*

In [13]:
hpt["_drg_payer"] = hpt.payer_name.apply(lambda p: normalize_payer(p, aliases))
m = hpt[(hpt._ein=="131740114") & (hpt._code=="872") & (hpt._drg_payer=="aetna")]
print("HPT Montefiore Aetna 872 plans/dollars:")
print(m[["plan_name","_dollar"]].to_string(index=False))
tic_aetna_872 = tic[(tic._code=="872") & (tic._payer=="aetna")]
print("\nTiC rows for Aetna DRG 872 (any hospital):", len(tic_aetna_872), "  <-- the gap")

HPT Montefiore Aetna 872 plans/dollars:
 plan_name  _dollar
       ASA 51350.81
  Medicare 13434.30
Commercial 45907.79

TiC rows for Aetna DRG 872 (any hospital): 6   <-- the gap


**Read this:** the hospital lists Aetna 872; the payer file has **none** — so it can only ever be a one-sided (`hpt_only`) row. That asymmetry (commercial DRG rates present on the hospital side, sparse on the payer side) is a structural feature of these files, not a bug.

## 9. Fill-rate snapshot

In [14]:
def fill(df):
    nn = df.apply(lambda c: c.astype(str).str.strip().replace({"nan":"","None":""}).ne("").mean())
    return (nn*100).round(0).astype(int)
print("TiC fill % (key cols):")
print(fill(tic[["payer","code","code_type","ein","billing_class","rate","cms_baseline_rate"]]).to_string())
print("\nHPT fill % (key cols):")
print(fill(hpt[["source_file_name","license_number","payer_name","plan_name","raw_code",
                "description","setting","standard_charge_negotiated_dollar",
                "standard_charge_negotiated_percentage","standard_charge_methodology"]]).to_string())

TiC fill % (key cols):
payer                100
code                 100
code_type            100
ein                  100
billing_class        100
rate                 100
cms_baseline_rate    100

HPT fill % (key cols):
source_file_name                         100
license_number                           100
payer_name                               100
plan_name                                100
raw_code                                 100
description                              100
setting                                  100
standard_charge_negotiated_dollar         86
standard_charge_negotiated_percentage     23
standard_charge_methodology               93


## Takeaways that drive the pipeline

1. **Key on filename-EIN, not `license_number`.**
2. **Normalize DRG code spellings** (`MS-DRG 872` → `872`).
3. **Resolve payer names** (and refuse to over-match — the long tail is where false merges hide).
4. **Infer HPT billing class**, but treat it as weak (it misroutes the $6,438).
5. **Drop/flag the un-comparable**: percentage rows, LOCAL chargemaster, no-dollar.
6. **Collapse each side to one row per `(ein, code, code_type, payer, billing_class)`**, preserving dispersion (min/max/count).
7. **Expect one-sided rows** (the Aetna-872 gap) and **rate-level agreement the key-join can miss** (the $6,438) — both handled in `pipeline.ipynb`.